# 🏀 篮球检测模型训练（YOLOv11）— Colab 一键版

**流程**：免费 T4 GPU → 下载已标注数据集（球+篮筐，5482 张）→ 训练 YOLOv11n → 导出权重

全程约 40–60 分钟，中间不用管。开始前准备好你的 **Roboflow API key**
（roboflow.com → 头像 → Settings → API Keys → Private API Key）。

## 0. 确认 GPU 已挂载（输出里要有 Tesla T4）
如果显示 `No devices`，点菜单 **Runtime（代码执行程序）→ Change runtime type（更改运行时类型）→ T4 GPU → Save**，再重跑。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q ultralytics roboflow

## 1. 输入 Roboflow API key（只存在当前会话，不会保存）

In [ ]:
import os
from getpass import getpass
os.environ["ROBOFLOW_API_KEY"] = getpass("粘贴 Roboflow API key: ")
print("key 已读入")

## 2. 下载数据集

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("sc-xqmxu").project("basketball-and-net-detection")
ds = project.version(7).download("yolov11", location="/content/basketball")
import yaml
with open("/content/basketball/data.yaml") as f:
    print("类别:", yaml.safe_load(f).get("names"))

## 3. 看一眼数据

In [ ]:
import yaml, glob
with open("/content/basketball/data.yaml") as f:
    print(yaml.safe_load(f))
for split in ("train", "valid", "test"):
    n = len(glob.glob(f"/content/basketball/{split}/images/*"))
    print(f"{split}: {n} 张")

## 4. 训练（约 40–60 分钟，跑完自动停）

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")  # COCO 预训练权重做起点，迁移学习
model.train(
    data="/content/basketball/data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,          # Colab T4
    name="ball_hoop",
    project="/content/runs",
)

## 5. 看训练结果

In [ ]:
from ultralytics import YOLO
model = YOLO("/content/runs/ball_hoop/weights/best.pt")
metrics = model.val(data="/content/basketball/data.yaml")
print(f"mAP50: {metrics.box.map50:.3f}  mAP50-95: {metrics.box.map:.3f}")
from IPython.display import Image
Image("/content/runs/ball_hoop/results.png")

## 6. 导出权重

In [ ]:
import shutil
from google.colab import files
shutil.copy("/content/runs/ball_hoop/weights/best.pt", "/content/ball_hoop.pt")
files.download("/content/ball_hoop.pt")  # 自动下载到你电脑
print("done")

**最后一步**：把下载到的 `ball_hoop.pt` 放进项目目录的 `weights/` 文件夹，
检测器会自动加载它（`detector.py` 会打印 `custom model` 即成功）。